In [ ]:
# Cell 1 — Install missing packages (Colab already has torch+CUDA, numpy, pandas, etc.)
!pip install -r requirements-colab.txt -q

In [ ]:
# Cell 2 — Mount Google Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

REPO_URL  = 'https://github.com/<your-username>/aerial-detection-production.git'
REPO_DIR  = '/content/aerial-detection-production'

result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
if result.returncode != 0 and 'already exists' not in result.stderr:
    raise RuntimeError(result.stderr)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import os
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3 — Imports, seeds, and MLflow env vars
# MLflow env vars MUST be set before model.train() —
# Ultralytics checks for MLflow on the first training step, not at import.
import os
os.environ['MLFLOW_TRACKING_URI']    = 'file:./mlruns'
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'aerial_yolov8'

import json
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import mlflow
from ultralytics import YOLO
from ultralytics.utils import yaml_load

from training.yolov8_train import (
    create_dataset_yaml,
    evaluate,
    export_metrics,
    sample_annotation_check,
    train,
    verify_dataset_structure,
)

random.seed(42)
np.random.seed(42)

print('Ultralytics and MLflow ready')

In [ ]:
# Cell 4 — Config block
RANDOM_STATE  = 42
MODEL_NAME    = 'yolov8n.pt'   # nano: 3.2M params — appropriate for 3K-image dataset
IMGSZ         = 640            # YOLOv8 standard (separate from classification 224×224)
EPOCHS        = 50
BATCH         = 16
FREEZE_LAYERS = 3              # confirmed after checking nano backbone layer count in Cell 6
PATIENCE      = 10
CONFIDENCE_THR = 0.5
DEVICE        = 0              # T4 GPU
PROJECT_DIR   = 'runs/detect'
RUN_NAME      = 'aerial_nano'

DRIVE_ROOT    = Path('/content/drive/MyDrive/aerial_detection')
DATASET_DIR   = DRIVE_ROOT / 'object_detection_Dataset'
MODELS_DIR    = Path('models')
YAML_PATH     = Path('data/yolo_dataset.yaml')

MODELS_DIR.mkdir(exist_ok=True)
print(f'Dataset:   {DATASET_DIR}')
print(f'YAML:      {YAML_PATH}')
print(f'Model:     {MODEL_NAME} | imgsz={IMGSZ} | batch={BATCH} | freeze={FREEZE_LAYERS}')

In [ ]:
# Cell 5 — Verify dataset structure, annotation format, and create YAML

# Step 1: Directory structure check
print('=== Dataset Structure ===' )
report = verify_dataset_structure(str(DATASET_DIR))
for split, info in report.items():
    if 'error' in info:
        print(f'  {split}: ERROR — {info["error"]}')
    else:
        status = '✓' if info['n_unpaired'] == 0 else f'⚠ {info["n_unpaired"]} unpaired'
        print(f'  {split}: {info["n_images"]} images | {info["n_labels"]} labels | {status}')

# Step 2: Annotation format — manual class ID check before GPU time is spent
print('\n=== Sample Annotations (verify class_id 0=bird, 1=drone) ===')
sample_annotation_check(str(DATASET_DIR), n_samples=3)

# Step 3: Create YAML with absolute Colab Drive path
create_dataset_yaml(dataset_root=str(DATASET_DIR), output_path=str(YAML_PATH))

# Step 4: Verify Ultralytics resolves paths correctly
print('\n=== YAML Resolved Paths ===')
data = yaml_load(str(YAML_PATH))
print(json.dumps(data, indent=2, default=str))

In [ ]:
# Cell 6 — Load yolov8n.pt and print backbone layer count
# Confirms freeze=3 is appropriate (nano has ~9-10 backbone layers;
# freeze=10 would freeze the entire backbone, defeating fine-tuning).

base_model = YOLO(MODEL_NAME)

print('=== YOLOv8n Layer Summary ===')
print(f'Total model layers: {len(list(base_model.model.named_modules()))}')
print('\nFirst 15 named modules (backbone ends at SPPF):')
for i, (name, _) in enumerate(base_model.model.named_modules()):
    if i >= 15:
        break
    print(f'  {i:3d}  {name}')

print(f'\nfreeze={FREEZE_LAYERS} → freezing first {FREEZE_LAYERS} backbone layers only')
print(f'Params (trainable): {sum(p.numel() for p in base_model.model.parameters() if p.requires_grad):,}')

del base_model  # free memory before training

In [ ]:
# Cell 7 — Fine-tune YOLOv8n
# MLflow env vars already set in Cell 3. Ultralytics auto-detects and logs:
#   per-epoch: box_loss, cls_loss, dfl_loss, mAP50, mAP50-95
#   artifacts: best.pt, last.pt, training plots

mlflow.set_experiment('aerial_yolov8')

model, elapsed_min = train(
    yaml_path     = str(YAML_PATH),
    model_name    = MODEL_NAME,
    epochs        = EPOCHS,
    imgsz         = IMGSZ,
    batch         = BATCH,
    freeze        = FREEZE_LAYERS,
    patience      = PATIENCE,
    device        = DEVICE,
    seed          = RANDOM_STATE,
    project       = PROJECT_DIR,
    name          = RUN_NAME,
)

# Best weights location
BEST_WEIGHTS = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'best.pt'
print(f'\nTraining complete in {elapsed_min:.1f} min')
print(f'Best weights: {BEST_WEIGHTS}  ({BEST_WEIGHTS.stat().st_size / 1e6:.1f} MB)')

In [ ]:
# Cell 8 — Training curves
# Ultralytics saves results.csv and plots in the run directory.
import pandas as pd

results_csv = Path(PROJECT_DIR) / RUN_NAME / 'results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()  # strip whitespace from column names

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('YOLOv8n Training Curves — Aerial Detection', fontsize=14)

plot_cols = [
    ('train/box_loss',    'Train Box Loss'),
    ('train/cls_loss',    'Train Cls Loss'),
    ('train/dfl_loss',    'Train DFL Loss'),
    ('metrics/mAP50(B)',  'mAP@0.5'),
    ('metrics/mAP50-95(B)', 'mAP@0.5:0.95'),
    ('val/box_loss',      'Val Box Loss'),
]

for ax, (col, title) in zip(axes.flat, plot_cols):
    if col in df.columns:
        ax.plot(df['epoch'], df[col], linewidth=2)
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f'{title} (column not found)')

plt.tight_layout()
plt.savefig(MODELS_DIR / 'yolov8_training_curves.png', dpi=150)
plt.show()
print(f'Epochs run: {len(df)} / {EPOCHS} (early_stopped={len(df) < EPOCHS})')

In [ ]:
# Cell 9 — Evaluate on test split
# Load best weights for evaluation (not last epoch weights)
best_model = YOLO(str(BEST_WEIGHTS))

test_metrics = evaluate(best_model, yaml_path=str(YAML_PATH), split='test', conf=CONFIDENCE_THR)

print('=== Test Set Metrics ===')
print(f'  mAP@0.5:      {test_metrics["map50"]:.4f}')
print(f'  mAP@0.5:0.95: {test_metrics["map50_95"]:.4f}')
print(f'  Precision:    {test_metrics["precision"]:.4f}')
print(f'  Recall:       {test_metrics["recall"]:.4f}')
print(f'  Conf thresh:  {test_metrics["confidence_threshold"]}')

In [ ]:
# Cell 10 — Inference on 6 test images with bounding box overlay
# result.plot() returns a BGR numpy array (OpenCV convention) — must convert to RGB for matplotlib.

test_img_dir = DATASET_DIR / 'test' / 'images'
test_imgs    = sorted(test_img_dir.glob('*.jpg'))[:6]

if not test_imgs:
    print('No test images found — skipping inference preview')
else:
    results = best_model.predict(
        source  = [str(p) for p in test_imgs],
        conf    = CONFIDENCE_THR,
        save    = False,
        verbose = False,
    )

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('YOLOv8n Inference — Aerial Test Images', fontsize=13)

    for ax, result in zip(axes.flat, results):
        # result.plot() → BGR numpy array with drawn boxes
        img_bgr = result.plot()
        img_rgb = img_bgr[:, :, ::-1]  # BGR → RGB for matplotlib
        ax.imshow(img_rgb)
        ax.axis('off')
        if result.boxes is not None and len(result.boxes):
            labels = [
                f"{result.names[int(c)]} {conf:.2f}"
                for c, conf in zip(result.boxes.cls, result.boxes.conf)
            ]
            ax.set_title(', '.join(labels), fontsize=9)
        else:
            ax.set_title('No detections', fontsize=9)

    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'yolov8_inference_preview.png', dpi=150)
    plt.show()
    print('Saved: models/yolov8_inference_preview.png')

In [ ]:
# Cell 11 — Save metrics_yolov8.json and log artifact to MLflow
import pandas as pd

results_csv = Path(PROJECT_DIR) / RUN_NAME / 'results.csv'
df_r = pd.read_csv(results_csv)
epochs_run    = len(df_r)
early_stopped = epochs_run < EPOCHS

# Retrieve the MLflow run ID created by Ultralytics during training
runs = mlflow.search_runs(experiment_names=['aerial_yolov8'], order_by=['start_time DESC'])
mlflow_run_id = runs.iloc[0]['run_id'] if not runs.empty else 'unknown'

export_metrics(
    metrics            = test_metrics,
    training_time_min  = elapsed_min,
    epochs_run         = epochs_run,
    early_stopped      = early_stopped,
    mlflow_run_id      = mlflow_run_id,
    output_path        = str(MODELS_DIR / 'metrics_yolov8.json'),
)

# Log JSON artifact to the same MLflow run
with mlflow.start_run(run_id=mlflow_run_id):
    mlflow.log_artifact(str(MODELS_DIR / 'metrics_yolov8.json'))
    mlflow.log_artifact(str(MODELS_DIR / 'yolov8_inference_preview.png'))

with open(MODELS_DIR / 'metrics_yolov8.json') as f:
    print(json.dumps(json.load(f), indent=2))

## Section Comparison — Detection vs Classification

YOLOv8 and the classification models (Custom CNN, EfficientNetB0) **solve different problems**:

| | Custom CNN | EfficientNetB0 | YOLOv8n |
|---|---|---|---|
| **Task** | Whole-image classification | Whole-image classification | Object detection |
| **Output** | Single probability score | Single probability score | Bounding boxes + class scores |
| **Question answered** | Is there a drone in this image? | Is there a drone in this image? | Where is the drone and what is it? |
| **Primary metric** | Accuracy / F1 | Accuracy / F1 | mAP@0.5 |
| **Input size** | 224×224 | 224×224 | 640×640 |

**mAP@0.5 and accuracy/F1 are not directly comparable** — they measure performance on different problem formulations (localization vs. whole-image label).

These are **complementary capabilities**:
- FastAPI `/predict` endpoint uses EfficientNetB0 for fast, cache-first classification
- YOLOv8 adds spatial awareness — useful for multi-object scenes or drone tracking
- Both pipelines are independent and additive

In [ ]:
# Cell 13 — Copy best weights to Drive + local models/ + zip mlruns
drive_models = DRIVE_ROOT / 'models'
drive_models.mkdir(exist_ok=True)

# Best weights: saved by Ultralytics at {project}/{name}/weights/best.pt
BEST_WEIGHTS = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'best.pt'

# Copy to Drive (persistent across Colab sessions)
shutil.copy(BEST_WEIGHTS, drive_models / 'yolov8n_best.pt')

# Copy to local models/ (for Docker volume mount)
shutil.copy(BEST_WEIGHTS, MODELS_DIR / 'yolov8n_best.pt')

# Copy metrics and training curves to Drive
for fname in ('metrics_yolov8.json', 'yolov8_training_curves.png', 'yolov8_inference_preview.png'):
    src = MODELS_DIR / fname
    if src.exists():
        shutil.copy(src, drive_models / fname)

# Zip mlruns to Drive (same pattern as Sections 3 and 4)
shutil.make_archive(
    str(DRIVE_ROOT / 'mlruns_yolov8'),
    'zip',
    '/content/mlruns',
)

best_mb = BEST_WEIGHTS.stat().st_size / 1e6
print(f'Best weights:     {BEST_WEIGHTS}  ({best_mb:.1f} MB)')
print(f'Drive copy:       {drive_models / "yolov8n_best.pt"}')
print(f'Local copy:       {MODELS_DIR / "yolov8n_best.pt"}')
print(f'MLflow archive:   {DRIVE_ROOT / "mlruns_yolov8.zip"}')
print('\n=== Section 5 Complete ===')
print(f'mAP@0.5:      {test_metrics["map50"]:.4f}')
print(f'mAP@0.5:0.95: {test_metrics["map50_95"]:.4f}')
print(f'Precision:    {test_metrics["precision"]:.4f}')
print(f'Recall:       {test_metrics["recall"]:.4f}')